# Incident Extraction from Weekly Chat Exports

This notebook demonstrates how to:
1. Parse weekly chat exports (Claude, ChatGPT)
2. Extract incident mentions from conversations
3. Build timeline from chat data
4. Correlate with schedules and notes

## Use Case
Extract real-time conversations about incidents to build comprehensive timelines for legal or medical cases.

In [ ]:
import os
from pathlib import Path
from datetime import datetime
import json

from dotenv import load_dotenv

# Load API keys
load_dotenv()

# Import our ecosystem modules
import sys
sys.path.append(str(Path.cwd().parent.parent))

from research_evidence_ecosystem.mcp_servers.chat_timeline_mcp.weekly_chat_parser import WeeklyChatParser
from research_evidence_ecosystem.mcp_servers.chat_timeline_mcp.incident_extractor import IncidentExtractor

## Step 1: Parse Weekly Chat Exports

Point to your exported chat data organized by week.

In [ ]:
# Example: Parse Claude export for week 1 of 2026
parser = WeeklyChatParser()

# Path to your weekly chat export
# Format: See weekly_chat_parser.py for expected JSON structure
claude_export_path = "data/chat_exports/2026_week_01_claude.json"

# Parse the export
week_data = parser.parse_claude_export(
    export_path=claude_export_path,
    week_id="2026-W01"
)

print(f"Week: {week_data.week_id}")
print(f"Conversations: {len(week_data.conversations)}")
print(f"Total messages: {week_data.total_messages}")
print(f"Incident-related messages: {week_data.incidents_mentioned}")
print(f"\nTop entities mentioned: {week_data.entities_discussed[:10]}")

## Step 2: Extract Incidents from Conversations

Use Claude to extract structured incident mentions from each conversation.

In [ ]:
extractor = IncidentExtractor()

# Extract incidents from all conversations
all_mentions = []

for conversation in week_data.conversations:
    print(f"\nAnalyzing conversation: {conversation.primary_topic[:50]}...")
    
    # Extract incidents (specify domain for better extraction)
    mentions = await extractor.extract_incidents_from_conversation(
        conversation=conversation,
        domain="medical"  # or "legal" or "general"
    )
    
    print(f"  Found {len(mentions)} incident mentions")
    
    for mention in mentions:
        print(f"    - {mention.description[:60]}...")
        print(f"      Date: {mention.mentioned_date} ({mention.date_precision})")
        print(f"      Confidence: {mention.confidence:.2f}")
        print(f"      Firsthand: {mention.is_firsthand}")
    
    all_mentions.extend(mentions)

print(f"\nTotal incidents extracted: {len(all_mentions)}")

## Step 3: Cluster Related Mentions

Multiple conversations may reference the same incident. Cluster them together.

In [ ]:
# Cluster mentions into unified incidents
reconstructed_incidents = extractor.cluster_incidents(all_mentions)

print(f"Unique incidents: {len(reconstructed_incidents)}\n")

for incident in reconstructed_incidents:
    print(f"\n{'='*60}")
    print(f"Incident: {incident.description}")
    print(f"Type: {incident.incident_type}")
    print(f"Date: {incident.occurred_date} ({incident.date_precision})")
    print(f"Entities: {', '.join(incident.entities_involved)}")
    print(f"Confidence: {incident.overall_confidence:.2f}")
    print(f"Corroborated by {incident.corroboration_count} sources")
    print(f"\nMentions:")
    for mention in incident.mentions:
        print(f"  - {mention.excerpt[:80]}...")
        print(f"    Source: {mention.source_id}")
        print(f"    Timestamp: {mention.source_timestamp}")

## Step 4: Build Timeline

Create chronological timeline from extracted incidents.

In [ ]:
# Sort incidents by date
timeline_events = []
for incident in reconstructed_incidents:
    if incident.occurred_date != "unknown":
        timeline_events.append({
            "date": incident.occurred_date,
            "precision": incident.date_precision,
            "type": incident.incident_type,
            "description": incident.description,
            "confidence": incident.overall_confidence,
            "sources": len(incident.mentions)
        })

# Sort chronologically
timeline_events.sort(key=lambda e: str(e["date"]))

print("\n" + "="*60)
print("TIMELINE")
print("="*60 + "\n")

for event in timeline_events:
    print(f"{event['date']} ({event['precision']})")
    print(f"  {event['description']}")
    print(f"  Type: {event['type']} | Confidence: {event['confidence']:.2f} | Sources: {event['sources']}")
    print()

## Step 5: Export for Obsidian

Create markdown notes for each incident with backlinks.

In [ ]:
# Export to Obsidian-compatible markdown
output_dir = Path("obsidian_export/incidents")
output_dir.mkdir(parents=True, exist_ok=True)

for incident in reconstructed_incidents:
    # Create filename from incident type and date
    safe_date = str(incident.occurred_date).replace(":", "-")[:10]
    filename = f"{safe_date}_{incident.incident_type}_{incident.incident_id[:8]}.md"
    
    # Build markdown content
    md_content = f"""---
type: incident
date: {incident.occurred_date}
date_precision: {incident.date_precision}
incident_type: {incident.incident_type}
confidence: {incident.overall_confidence}
sources: {incident.corroboration_count}
tags: [incident, {incident.incident_type}]
---

# {incident.description}

## Details
- **Date**: {incident.occurred_date} ({incident.date_precision})
- **Type**: {incident.incident_type}
- **Confidence**: {incident.overall_confidence:.2f}
- **Corroboration**: {incident.corroboration_count} sources

## Entities Involved
{chr(10).join(f'- [[{entity}]]' for entity in incident.entities_involved)}

## Source Mentions
"""
    
    for i, mention in enumerate(incident.mentions, 1):
        md_content += f"""
### Mention {i}
- **Source**: {mention.source_id}
- **Date**: {mention.source_timestamp}
- **Firsthand**: {mention.is_firsthand}
- **Confidence**: {mention.confidence:.2f}

> {mention.excerpt}
"""
    
    # Write to file
    with open(output_dir / filename, "w") as f:
        f.write(md_content)
    
    print(f"Created: {filename}")

print(f"\nExported {len(reconstructed_incidents)} incident notes to {output_dir}")

## Step 6: Correlate with Schedule (Optional)

If you have calendar data, match incidents to scheduled appointments.

In [ ]:
# Example: Load schedule data (iCal, Google Calendar export, etc.)
# This would use the schedule_mcp module

# For now, manual correlation example:
sample_appointments = [
    {"date": "2026-01-06", "type": "doctor_visit", "title": "Dr. Smith Checkup"},
    {"date": "2026-01-08", "type": "procedure", "title": "MRI Scan"},
]

print("\nSchedule Correlations:\n")

for incident in reconstructed_incidents:
    incident_date = str(incident.occurred_date)[:10]
    
    for appt in sample_appointments:
        # Check if incident date is close to appointment
        if appt["date"] in str(incident_date):
            print(f"✓ Incident '{incident.description[:40]}...'")
            print(f"  Correlates with: {appt['title']} on {appt['date']}")
            print()

## Summary

You now have:
- ✅ Parsed weekly chat exports
- ✅ Extracted structured incident mentions
- ✅ Clustered related mentions
- ✅ Built chronological timeline
- ✅ Exported to Obsidian
- ✅ Correlated with schedule data

## Next Steps

1. **Process multiple weeks** - Run this for all your weekly exports
2. **Add schedule integration** - Parse iCal/Google Calendar
3. **Integrate notes** - Add prior research, medical records, legal documents
4. **Build knowledge graph** - Visualize entity relationships in Obsidian
5. **Verify with Perplexity** - Cross-check facts with web research